# 00 - Variables del Workshop BNCR

Ejecute **todas las celdas** antes de continuar con `00_setup`.


In [0]:
catalog_name = "BNS"
schema_raw = "raw"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"
volume = "transacciones"

vol_path = f"/Volumes/{catalog_name}/{schema_raw}/{volume}"
repo_url = "https://github.com/Ricojacob01/Latam_resources_spanish"
repo_path = "Data_Engineering"

print(f"Catalogo   : {catalog_name}")
print(f"Volumen    : {vol_path}")
print(f"Repositorio: {repo_url}/{repo_path}")


In [0]:
def _notebook_path():
    import urllib.parse
    raw = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    return urllib.parse.unquote(raw)


def _ruta_archivos_workspace(carga):
    notebook_path = _notebook_path()
    marker = "/Data_Engineering"
    if marker not in notebook_path:
        return None
    rel = notebook_path.split(marker)[0] + f"{marker}/Files/{carga}"
    return "/Workspace" + rel


def _copiar_desde_workspace(carga):
    src = _ruta_archivos_workspace(carga)
    if not src:
        return False
    try:
        items = dbutils.fs.ls(src)
    except Exception:
        return False
    for item in items:
        name = item.name.rstrip("/")
        dst = f"{vol_path}/{name}"
        try:
            dbutils.fs.rm(dst, recurse=True)
        except Exception:
            pass
        dbutils.fs.cp(item.path, dst, recurse=item.isDir())
    return True


def _copiar_desde_github(carga):
    import json
    import urllib.request
    from urllib.parse import quote
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "bncr-workshop"}

    def copiar_rama(api_path, dest):
        encoded = quote(api_path, safe="/")
        url = f"https://api.github.com/repos/Ricojacob01/Latam_resources_spanish/contents/{encoded}"
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=90) as resp:
            items = json.loads(resp.read().decode())
        for item in items:
            name = item["name"]
            target = f"{dest}/{name}"
            if item["type"] == "file":
                with urllib.request.urlopen(item["download_url"], timeout=90) as f:
                    dbutils.fs.put(target, f.read().decode("utf-8"), overwrite=True)
            else:
                dbutils.fs.mkdirs(target)
                copiar_rama(item["path"], target)

    copiar_rama(f"Data_Engineering/Files/{carga}", vol_path)


def _resumen_archivos(path):
    filas = []

    def recorrer(p):
        for item in dbutils.fs.ls(p):
            if item.isDir():
                recorrer(item.path)
            else:
                filas.append({
                    "archivo": item.name,
                    "ruta": item.path.replace("dbfs:", ""),
                    "bytes": int(item.size),
                })

    recorrer(path)
    return filas


def carga_datos(carga):
    if carga not in ("initial", "incremental"):
        raise ValueError("carga debe ser 'initial' o 'incremental'")

    esperados = {"initial": 7, "incremental": 3}
    dbutils.fs.mkdirs(vol_path)

    if _copiar_desde_workspace(carga):
        metodo = "Git folder del workspace"
    else:
        _copiar_desde_github(carga)
        metodo = "GitHub API"

    filas = _resumen_archivos(vol_path)
    total_bytes = sum(f["bytes"] for f in filas)
    print(f"Carga '{carga}' completada ({metodo})")
    print(f"Archivos en volumen: {len(filas)} (esperados: {esperados[carga]}) | Total: {total_bytes:,} bytes")

    if len(filas) < esperados[carga]:
        raise RuntimeError(
            f"Solo se encontraron {len(filas)} archivos en {vol_path}. "
            "Haga Git Pull y vuelva a ejecutar esta celda."
        )

    display(spark.createDataFrame(filas))
